# FBI Crime Data API - Analyse

Dieses Notebook analysiert Kriminaldaten von der FBI Crime Data API und stellt sie mit Plotly dar.

## 1. Einführung in die FBI Crime Data API 

Die FBI Crime Data API ist eine API (bracht API-KEY) der Federal Bureau of Investigation, die Kriminaldaten aus Californien bereitstellt. Wollte es zuerst über die ganze USA machen, was sicher interessanter wäre, jedoch waren die Server meist überlastet und die Daten-Packs einfach riesig :) 

### Was liefert die API?
- Entwicklung über die Zeit 
- Durchschnitt pro Jahr
- Höchste und niedrigste Werte pro Jahr
- Durchschnitt nach Monaten

Die Rate beschreibt die Anzahl der Fälle pro 100.000 Einwohner. Dadurch lassen sich Kriminalitätswerte besser vergleichen als mit reinen Fallzahlen.

### API Verbindung und Links zu den Quellen
- **Base URL:** "https://api.usa.gov/crime/fbi/cde"
- **Endpoint:** "https://api.usa.gov/crime/fbi/cde/summarized/state/CA/aggravated-assault"
- **Offizielle FBI Crime DATA Explorer:** "https://cde.ucr.cjis.gov/LATEST/webapp/#/pages/home"
- **Offizielle Seite FBI Goverment:** "https://www.fbi.gov/how-we-can-help-you/more-fbi-services-and-information/ucr"
- **Rückgabe:** JSON-Format mit Link "https://api.usa.gov/crime/fbi/cde/summarized/state/CA/aggravated-assault?from=01-2010&to=12-2022&API_KEY=InRFiFa9Q7o6pakyOJcWOJUowuwa5MmGJdZ18X1N"

**API-KEY:** Der Key für die URL muss unter folgenden Dokument beantragt werden, ich habe zb. gleich nach einen Werkstag per Mail zugesendet bekommen "https://cde.ucr.cjis.gov/LATEST/webapp/#/pages/docApi" dann unter Get an API-Key klicken und die Daten aussfüllen 

## 2. Bibliotheken importieren

In [95]:
import requests
import pandas as pd
import plotly.express as px

BASE_URL = "https://api.usa.gov/crime/fbi/cde"
print("Bibliotheken erfolgreich geladen")

Bibliotheken erfolgreich geladen


## 3. Daten von der FBI API abrufen

In [ ]:
import requests

API_KEY = " Habe meine API_Key aus Sicherheitsgründen entfernt, bitte selbst einen anfordern, Link steht oben(API_KEY), danke :) "

url = "https://api.usa.gov/crime/fbi/cde/summarized/state/CA/aggravated-assault"

params = {
    "from": "01-2010",
    "to": "12-2022",
    "API_KEY": API_KEY
}

response = requests.get(url, params=params)

print(response.status_code)

response.raise_for_status()
data = response.json()

200


## 4. Struktur der JSON-Antwort ansehen

In [97]:
print(data.keys())
print(data["offenses"].keys())
print(data["offenses"]["rates"].keys())

dict_keys(['offenses', 'tooltips', 'populations', 'cde_properties'])
dict_keys(['rates', 'actuals'])
dict_keys(['California Offenses', 'California Clearances', 'United States Offenses', 'United States Clearances'])


## 5. JSON-Daten in einen DataFrame umwandeln

In [98]:
rates = data["offenses"]["rates"]["California Offenses"]

df = pd.DataFrame(list(rates.items()), columns=["monat", "rate"])

df.head()

,monat,rate
0,01-2010,20.97
1,01-2011,19.86
2,01-2012,19.41
3,01-2013,18.38
4,01-2014,18.34


## 6. Daten vorbereiten

In [99]:
df["datum"] = pd.to_datetime(df["monat"], format="%m-%Y")
df["jahr"] = df["datum"].dt.year
df["rate"] = pd.to_numeric(df["rate"])

df = df.sort_values("datum").reset_index(drop=True)

df.head()

,monat,rate,datum,jahr
0,01-2010,20.97,2010-01-01,2010
1,02-2010,18.53,2010-02-01,2010
2,03-2010,21.08,2010-03-01,2010
3,04-2010,21.13,2010-04-01,2010
4,05-2010,23.68,2010-05-01,2010


## 7. Datenausgaben

### 7.1 Plot 1: Entwicklung über die Zeit 

In [100]:
diagramm_zeit = px.line(
    df,
    x="datum",
    y="rate",
    markers=True,
    title="Aggravated Assault Rate in Kalifornien von 2010 bis 2022",
    labels={
        "datum": "Datum",
        "rate": "Rate"
    }
)

diagramm_zeit.show()

## 7.2 Plot 2: Durchschnitt pro Jahr

In [101]:
df_jahr = df.groupby("jahr", as_index=False)["rate"].mean()

fig2 = px.bar(
    df_jahr,
    x="jahr",
    y="rate",
    title="Durchschnittliche Aggravated Assault Rate pro Jahr",
    labels={
        "jahr": "Jahr",
        "rate": "Durchschnittliche Rate"
    }
)

fig2.show()

### 7.3 Plot 3: Höchste und niedrigste Werte pro Jahr

In [102]:
df_min_max = df.groupby("jahr", as_index=False).agg(
    niedrigste_rate=("rate", "min"),
    hoechste_rate=("rate", "max")
)

fig3 = px.line(
    df_min_max,
    x="jahr",
    y=["niedrigste_rate", "hoechste_rate"],
    markers=True,
    title="Höchste und niedrigste Rate pro Jahr",
    labels={
        "jahr": "Jahr",
        "value": "Rate",
        "variable": "Wert"
    }
)

fig3.show()

### 7.4 Plot 4: Durchschnitt nach Monaten

In [103]:
monats_daten = df.copy()
monats_daten["monat"] = monats_daten["datum"].dt.month

monats_durchschnitt = monats_daten.groupby("monat", as_index=False)["rate"].mean()

diagramm_monat = px.line(
    monats_durchschnitt,
    x="monat",
    y="rate",
    markers=True,
    title="Durchschnittliche Rate nach Monat",
    labels={
        "monat": "Monat",
        "rate": "Durchschnittliche Rate"
    }
)

diagramm_monat.show()